# Stock market chart pattern recognition using deep learning 

In [1]:
pip install settrade_v2

Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install cassandra-driver

Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install schedule

Note: you may need to restart the kernel to use updated packages.


# 1. Collect Data(Settrade APi)

In [1]:
from settrade_v2 import Investor
import pandas as pd
import cassandra
import re
import os
import glob
import numpy as np
import matplotlib.pyplot as plt
from cassandra.cluster import Cluster
import requests
from fuzzywuzzy import fuzz
from collections import defaultdict
from fuzzywuzzy import process
from nltk.tokenize import word_tokenize
import requests
import schedule
import time
from cassandra.cluster import Cluster
from datetime import datetime

/opt/anaconda3/lib/python3.12/site-packages/fuzzywuzzy/fuzz.py:11: UserWarning: Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning
  warnings.warn('Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning')


In [2]:
cluster = Cluster(['127.0.0.1'])  # or 'localhost'
session = cluster.connect()

In [6]:
# กำหนดค่า API Credentials
investor = Investor(
    app_id="a0iOat7M4FrOrGxS",          # เปลี่ยนเป็น App ID ของคุณ
    app_secret="AJcSTgM8JP+N2Uzi6eBGFxnv7i0T6RPzTX+6FiTgfgnp",  # เปลี่ยนเป็น App Secret ของคุณ
    broker_id="SANDBOX",  # เปลี่ยนเป็น Broker ID ของคุณ
    app_code="SANDBOX"
    
)
market = investor.MarketData()


In [17]:
# กรณี Investor
market = investor.MarketData()


res = market.get_candlestick(
    symbol="PTT",
    interval="1d",
    limit=1,
    normalized=True,
)

print(res)

{'lastSequence': 0, 'time': [1742749200], 'open': [30.75], 'high': [34.5], 'low': [30.0], 'close': [34.0], 'volume': [4305200], 'value': [134233800.0]}


In [19]:
print(type(res))

<class 'dict'>


In [30]:
# 👉 เชื่อมต่อ Cassandra
cluster = Cluster(['127.0.0.1'])  # Localhost
session = cluster.connect()

# 👉 เลือกใช้ Keyspace
session.set_keyspace('stock_data')

# 👉 ตรวจสอบและสร้างตาราง (ถ้ายังไม่มี)
session.execute("""
    CREATE TABLE IF NOT EXISTS candlestick_data (
        symbol TEXT,
        time TIMESTAMP PRIMARY KEY,
        open_price FLOAT,
        high_price FLOAT,
        low_price FLOAT,
        close_price FLOAT,
        volume INT,
        value FLOAT
    )
""")
print("✅ Keyspace และ Table พร้อมใช้งาน!")

# 👉 ฟังก์ชันดึงข้อมูลหุ้น
def fetch_and_store_stock(symbol="PTT", interval="1d", limit=10):
    res = market.get_candlestick(symbol=symbol, interval=interval, limit=limit, normalized=True)

    if not res:
        print(f"⚠️ ไม่พบข้อมูลสำหรับ {symbol}")
        return

    for i in range(len(res["time"])):
        timestamp = datetime.fromtimestamp(res["time"][i])  
        open_price = res["open"][i]
        high_price = res["high"][i]
        low_price = res["low"][i]
        close_price = res["close"][i]
        volume = res["volume"][i]
        value = res["value"][i]

        session.execute("""
            INSERT INTO candlestick_data (symbol, time, open_price, high_price, low_price, close_price, volume, value)
            VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
        """, (symbol, timestamp, open_price, high_price, low_price, close_price, volume, value))

    print(f"✅ เพิ่มข้อมูล {len(res['time'])} รายการของหุ้น {symbol} สำเร็จ!")

# 👉 ทดสอบดึงข้อมูลหุ้น BBL
fetch_and_store_stock(symbol="PTT", interval="1d", limit=10)

✅ Keyspace และ Table พร้อมใช้งาน!
✅ เพิ่มข้อมูล 10 รายการของหุ้น PTT สำเร็จ!


In [24]:
import time
from datetime import datetime
from cassandra.cluster import Cluster

# ✅ รายชื่อหุ้นที่ต้องการดึงข้อมูล (กำหนดเอง)
symbols = ["PTT", "AOT", "SCB", "CPALL", "ADVANC"]

# ✅ ฟังก์ชันที่ดึงข้อมูล Financial Data สำหรับหุ้นแต่ละตัว
def insert_financial_data(symbol):
    try:
        # 👉 ดึงข้อมูลจาก Settrade API หรือแหล่งข้อมูลอื่น ๆ
        res = market.get_candlestick(symbol=symbol, interval="1d", limit=10, normalized=True)

        if not res:
            print(f"⚠️ ไม่พบข้อมูลสำหรับ {symbol}")
            return

        # 👉 เชื่อมต่อ Cassandra (หรือฐานข้อมูลอื่น ๆ)
        cluster = Cluster(['127.0.0.1'])  # ถ้าเป็น localhost
        session = cluster.connect()
        session.set_keyspace('stock_data')

        # 👉 วนลูปเก็บข้อมูล
        for i in range(len(res["time"])):
            timestamp = datetime.fromtimestamp(res["time"][i])  
            open_price = res["open"][i]
            high_price = res["high"][i]
            low_price = res["low"][i]
            close_price = res["close"][i]
            volume = res["volume"][i]
            value = res["value"][i]

            # 👉 แทรกข้อมูลลงในตาราง
            session.execute("""
                INSERT INTO candlestick_data (symbol, time, open_price, high_price, low_price, close_price, volume, value)
                VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
            """, (symbol, timestamp, open_price, high_price, low_price, close_price, volume, value))

        print(f"✅ เพิ่มข้อมูล {len(res['time'])} รายการของหุ้น {symbol} สำเร็จ!")

    except Exception as e:
        print(f"❌ เกิดข้อผิดพลาดในการดึงข้อมูลของหุ้น {symbol}: {e}")

# ✅ ดึงข้อมูล Financial Data สำหรับหุ้นทุกตัว
for symbol in symbols:
    insert_financial_data(symbol)
    time.sleep(2)  # ⏳ ควบคุม API Rate Limit


✅ เพิ่มข้อมูล 10 รายการของหุ้น PTT สำเร็จ!
✅ เพิ่มข้อมูล 10 รายการของหุ้น AOT สำเร็จ!
✅ เพิ่มข้อมูล 10 รายการของหุ้น SCB สำเร็จ!
✅ เพิ่มข้อมูล 10 รายการของหุ้น CPALL สำเร็จ!
✅ เพิ่มข้อมูล 10 รายการของหุ้น ADVANC สำเร็จ!


In [32]:
from cassandra.cluster import Cluster
import pandas as pd

# เชื่อมต่อกับ Cassandra
cluster = Cluster(['127.0.0.1'])  # ระบุ IP ของ Cassandra node ที่คุณใช้งาน
session = cluster.connect('stock_data')  # เปลี่ยนเป็น keyspace ของคุณ

# สั่งเลือกข้อมูลจากตาราง candlestick_data
symbol = "PTT"
query = f"SELECT time, open_price, high_price, low_price, close_price FROM candlestick_data WHERE symbol = '{symbol}' ALLOW FILTERING;"

# ดึงข้อมูลจาก Cassandra
rows = session.execute(query)

# สร้าง DataFrame จากผลลัพธ์
data = []
for row in rows:
    data.append({
        "time": row.time,
        "open_price": row.open_price,
        "high_price": row.high_price,
        "low_price": row.low_price,
        "close_price": row.close_price
    })

df = pd.DataFrame(data)

# แสดง DataFrame
print(df)

        time  open_price  high_price  low_price  close_price
0 2025-03-24       30.75       34.50      30.00        34.00
1 2025-03-13       27.25       27.75      27.00        27.25
2 2025-03-19       29.50       30.00      29.25        29.75
3 2025-03-17       29.00       29.50      28.50        29.50
4 2025-03-11       28.00       28.25      27.00        28.00
5 2025-03-21       30.75       38.00      30.50        31.25
6 2025-03-20       33.50       33.50      31.00        31.50
7 2025-03-14       27.50       29.25      27.25        29.00
8 2025-03-12       28.00       28.25      27.25        27.25
9 2025-03-18       29.75       30.00      29.00        29.50
